In [4]:
import re
import pandas as pd

MRKH→RSTI, EONR→UPRO, RUALR→RUAL, EPLN→LEAS, MAIL→VKCO, LNTA→LENT, YNDX→YDEX, HHRU→HEAD, TCSG→T, AGRO→RAGR, FIVE→X5

In [5]:
FILE="stock-index-base-moex-rts-18122012-nowadays.xlsx"
OUT="moex_weights_panel.parquet"

xls=pd.ExcelFile(FILE)
sheets=[s for s in xls.sheet_names if re.fullmatch(r"\d{2}\.\d{2}\.\d{4}",s)]

rows=[]

for sheet in sheets:
    df=pd.read_excel(FILE,sheet_name=sheet,header=None)
    header=df.index[df.apply(lambda x:x.astype(str).str.strip().eq("Code").any(),axis=1)]
    if len(header)==0:continue
    h=header[0]
    df=pd.read_excel(FILE,sheet_name=sheet,header=h)
    code_col=next(c for c in df.columns if str(c).strip()=="Code")
    weight_col=next(c for c in df.columns if str(c).strip().startswith("Weight"))
    x=df[[code_col,weight_col]].copy()
    x.columns=["ticker","weight"]
    x=x.dropna(subset=["ticker","weight"])
    x["ticker"]=x["ticker"].astype(str).str.strip()
    x=x[x["ticker"].str.match(r"^[A-Z0-9]+$")]
    x["weight"]=x["weight"].apply(lambda v:float(str(v).replace("%","").replace(",",".").strip())/100 if isinstance(v,str) and "%" in v else float(str(v).replace(",",".")))
    x["date"]=pd.to_datetime(sheet,format="%d.%m.%Y")
    rows.append(x)

panel=pd.concat(rows).pivot_table(index="date",columns="ticker",values="weight",aggfunc="first").sort_index()
panel.columns.name=None
panel.to_parquet(OUT,engine="pyarrow")
print(panel)
print(panel.shape)
print(f"Saved: {OUT}")

                AFKS      AFLT      AGRO      AKRN      ALRS      ASTR  \
date                                                                     
2012-12-18  0.006200  0.003300       NaN  0.001500  0.003300       NaN   
2013-03-18  0.006200  0.003900       NaN  0.001500  0.004500       NaN   
2013-06-18  0.015500  0.003100       NaN  0.001100  0.003500       NaN   
2013-09-17  0.016300  0.002700       NaN       NaN  0.003400       NaN   
2013-12-17  0.020300  0.003200       NaN       NaN  0.008800       NaN   
2014-03-18  0.021700  0.003900       NaN       NaN  0.010000       NaN   
2014-06-17  0.023300  0.003000       NaN       NaN  0.010800       NaN   
2014-09-16  0.019700  0.002500       NaN  0.001100  0.010800       NaN   
2014-12-16  0.006300  0.002100       NaN  0.001300  0.011300       NaN   
2015-01-20  0.006300  0.002100       NaN  0.001300  0.011300       NaN   
2015-03-17  0.007700  0.001700       NaN  0.001900  0.014000       NaN   
2015-06-16  0.007800  0.001900       N

In [9]:
d = pd.read_parquet('Raw_data/YNDX_1h.parquet')
d.head()

,ticker,begin,open,high,low,close,volume,value,end
0,YNDX,2014-06-04 10:00:00,1546.7,1546.8,1546.7,1546.8,5001,7735546.7,2014-06-04 10:48:06
1,YNDX,2014-06-04 11:00:00,1546.8,1546.8,1545.0,1546.8,1902,2941102.0,2014-06-04 11:56:10
2,YNDX,2014-06-04 12:00:00,1546.8,1546.8,1546.8,1546.8,440,680592.0,2014-06-04 12:48:21
3,YNDX,2014-06-04 13:00:00,1546.8,1546.8,1546.8,1546.8,289,447025.2,2014-06-04 13:45:06
4,YNDX,2014-06-04 14:00:00,1546.8,1546.8,1546.8,1546.8,67,103635.6,2014-06-04 14:55:16


In [ ]:
from pathlib import Path

ROOT = Path('/Users/stepansapunkov/HMM_proj')
FIELDS = ['open', 'high', 'low', 'close', 'volume', 'value']
RENAMED = {'MRKH': 'RSTI', 'EONR': 'UPRO', 'RUALR': 'RUAL', 'EPLN': 'LEAS',
           'MAIL': 'VKCO', 'LNTA': 'LENT', 'YNDX': 'YDEX', 'HHRU': 'HEAD',
           'TCSG': 'T', 'AGRO': 'RAGR', 'FIVE': 'X5'}
IMOEX_FILE = ROOT / 'Raw_Data' / 'IMOEX_1h.parquet'
FILES = sorted(file for file in (ROOT / 'Raw_Data').glob('*_1h.parquet')
               if file != IMOEX_FILE)

In [ ]:
def load_hourly(file):
    old = file.name.split('_1h', 1)[0]
    new = RENAMED.get(old, old)
    return pd.read_parquet(file, columns=['begin', *FIELDS]).assign(ticker=new, current=old == new)

raw = (pd.concat(map(load_hourly, FILES), ignore_index=True).sort_values('current')
       .drop_duplicates(['begin', 'ticker'], keep='last'))  
tickers = sorted(raw['ticker'].unique())
assert 'IMOEX' not in tickers, 'IMOEX is a benchmark index, not a constituent security'
dataset = raw.set_index(['begin', 'ticker'])[FIELDS].unstack('ticker').swaplevel(axis=1)
dataset = dataset.reindex(columns=pd.MultiIndex.from_product([tickers, FIELDS])).sort_index()
dataset.columns = [f'{ticker}_{field}' for ticker, field in dataset.columns]
dataset.index.name = 'date'
dataset.to_parquet(ROOT / 'Dataset.parquet')
dataset.shape

In [ ]:
weights = pd.read_parquet(ROOT / 'moex_weights_panel.parquet')
weights.columns = [RENAMED.get(ticker, ticker) for ticker in weights]
weights = weights.T.groupby(level=0).sum(min_count=1).T  
membership = (weights.gt(0).reindex(columns=tickers, fill_value=False)
              .reindex(dataset.index, method='ffill').fillna(False).astype(bool))
membership.index.name = 'date'
membership.to_parquet(ROOT / 'membership.parquet')
membership.shape